# text-curation v1.6.0

Deterministic text curation and corpus compilation infrastructure.

This notebook demonstrates:

Core Features (v1.0–v1.5):
- Profile-driven curation
- Deterministic block pipelines
- Dataset-level exact deduplication
- Explicit row filtering

New in v1.6:
- Signal-only analysis blocks
- SHA-256 fingerprinting
- Deterministic MinHash deduplication
- Exact n-gram decontamination
- Dataset manifest
- Pipeline configuration hashing

All operations are deterministic and reproducible.

In [1]:
from datasets import Dataset

from text_curation import TextCurator
from text_curation.registry import get_profile
from text_curation.core.reproducibility import compute_pipeline_hash

# Stable dataset utilities
from text_curation.datasets import (
    deduplicate_exact,
    filter_rows,
)

# Advanced dataset utilities (v1.6)
from text_curation.datasets.advanced import (
    deduplicate_by_hash,
    minhash_deduplicate,
    decontaminate,
)

from text_curation.reports.manifest import DatasetManifest

c:\Users\patil\Documents\Portfolio\text-curation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Phase A — Core Curation

In [2]:
dataset = Dataset.from_dict({
    "text": [
        "This is a test document.",
        "This is a test document.",
        "Completely different content here.",
        ""
    ]
})

dataset

Dataset({
    features: ['text'],
    num_rows: 4
})

In [3]:
curator = TextCurator.from_profile(
    "web_pretrain_v1",
    collect_reports=True
)

processed = dataset.map(curator, batched=True)

processed

Map: 100%|██████████| 4/4 [00:00<00:00, 159.04 examples/s]


Dataset({
    features: ['text', 'curation_report'],
    num_rows: 4
})

In [4]:
processed["curation_report"][0]

{'block_stats': {},
 'blocks': ['RedactionBlock',
  'NormalizationBlock',
  'CodeSafeFormattingBlock',
  'ParagraphFormattingBlock',
  'BasicStructureBlock',
  'QualitySignalBlock',
  'TokenStatsBlock',
  'FingerprintBlock'],
 'extras': {},
 'input_stats': {'chars': 24, 'lines': 1, 'paragraphs': 1, 'words': 5},
 'output_stats': {'chars': 24, 'lines': 1, 'paragraphs': 1, 'words': 5},
 'profile_id': 'web_pretrain_v1',
 'signals_summary': {'avg_sentence_length': 1,
  'char_entropy': 1,
  'is_all_caps': 1,
  'is_blank': 1,
  'is_boilerplate_candidate': 1,
  'is_bullet': 1,
  'is_header': 1,
  'is_list_block': 1,
  'is_numbered_item': 1,
  'is_short': 1,
  'max_token_length': 1,
  'rare_token_ratio': 1,
  'repetition_count': 2,
  'repetition_score': 1,
  'sha256': 1,
  'stopword_ratio': 1,
  'token_count': 1,
  'unique_token_count': 1,
  'url_density': 1}}

Each document emits deterministic signals:

- Structural signals  
- Quality metrics  
- Token statistics  
- SHA-256 fingerprint  

No filtering is implicit.

# Phase B — Stable Dataset Utilities (v1.0–v1.5)

In [5]:
dedup_exact_ds, report_exact = deduplicate_exact(
    dataset,
    column="text",
)

dedup_exact_ds

Dataset({
    features: ['text'],
    num_rows: 3
})

In [6]:
filtered_ds, filter_report_dict = filter_rows(
    dataset,
    predicate=lambda row: len(row["text"].strip()) > 0,
    description="Remove empty documents",
)

filtered_ds

Filter (num_proc=1): 100%|██████████| 4/4 [00:03<00:00,  1.11 examples/s]


Dataset({
    features: ['text'],
    num_rows: 3
})

Filtering requires:

- Explicit predicate  
- Explicit description  
- Deterministic execution  

# Phase C — Advanced Deterministic Compilation (v1.6)

In [7]:
dedup_hash_ds, hash_report = deduplicate_by_hash(
    processed,
    column="text",
)

dedup_hash_ds

Dataset({
    features: ['text', 'curation_report'],
    num_rows: 3
})

In [8]:
dedup_minhash_ds, minhash_report = minhash_deduplicate(
    processed,
    column="text",
    ngram_size=2,
    num_hashes=10,
    threshold=0.9,
    seed=42,
)

dedup_minhash_ds

Dataset({
    features: ['text', 'curation_report'],
    num_rows: 3
})

MinHash properties:

- Explicit seed  
- Deterministic cluster formation  
- Canonical representative = lowest index  
- Order preserved  

In [9]:
benchmark = {"is a", "a test"}

decontaminated_ds, decon_report = decontaminate(
    processed,
    column="text",
    benchmark_ngrams=benchmark,
    ngram_size=2,
)

decontaminated_ds

Dataset({
    features: ['text', 'curation_report', 'overlap_score'],
    num_rows: 4
})

Decontamination computes overlap scores.

It does not filter automatically.

Detection and filtering are separate concerns.

# Phase D — Reproducibility & Lineage

In [10]:
profile = get_profile("web_pretrain_v1")
pipeline_hash = compute_pipeline_hash(profile)
pipeline_hash

'ad7814c68cbc889192c9199d53fb99f916d3436808be186fad7c0e5184bc63b9'

In [11]:
run1 = dataset.map(curator, batched=True)
run2 = dataset.map(curator, batched=True)

run1 == run2

Map: 100%|██████████| 4/4 [00:00<00:00, 443.71 examples/s]


False

This must return:

True

If it does, the pipeline is fully deterministic.

In [12]:
manifest = DatasetManifest.from_dataset(
    dataset=dedup_hash_ds,
    text_column="text",
    profile_ids=["web_pretrain_v1"],
    library_version="1.6.0",
    block_order=[b.__class__.__name__ for b in profile.blocks],
    total_token_count=sum(len(t.split()) for t in dedup_hash_ds["text"]),
    timestamp="2024-01-01T00:00:00Z",
)

manifest.to_dict()

{'profile_ids': ['web_pretrain_v1'],
 'library_version': '1.6.0',
 'block_order': ['RedactionBlock',
  'NormalizationBlock',
  'CodeSafeFormattingBlock',
  'ParagraphFormattingBlock',
  'BasicStructureBlock',
  'QualitySignalBlock',
  'TokenStatsBlock',
  'FingerprintBlock'],
 'dataset_hash': '76451870ce68e0a825a3ab888af0a4d8d18e4a990d0506f71c2d6b0f8aa37fde',
 'total_token_count': 9,
 'timestamp': '2024-01-01T00:00:00Z',
 'metadata': {}}

The manifest encodes:

- Profile IDs  
- Library version  
- Dataset hash  
- Token count  
- Block order  
- Timestamp  

This enables long-term corpus lineage tracking.

# Summary

text-curation v1.6.0 provides:

## Stable Core
- Deterministic pipelines  
- Explicit profiles  
- Exact deduplication  
- Explicit filtering  

## Advanced Compilation
- Signal-only analysis  
- Fingerprinting  
- Deterministic MinHash  
- Decontamination  
- Manifest lineage  

All without hidden randomness.

---

Before release:

- Restart kernel  
- Run all cells top-to-bottom  
- Ensure no nondeterministic output  
- Confirm replay test returns True  

Then v1.6.0 is safe to release.